In [4]:
!pip -q install -U \langchain \ langchain-google-genai \ langchain-community \ langchain-text-splitters \ google-genai \ pypdf \ faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.2/382.2 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/

- LangChain-Google-GenAI - since we are using LLM from Google-Gemini
- Text-Splitters - helpful when we perform chunking operation
- PyPDF - since our of source of data is a PDF file - to load and read the data
- FAISS is a Vector Store which we will use when we build our RAG Pipeline

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI #we will configure our LLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [7]:
import os

os.environ["GOOGLE_API_KEY"]="-"

llm=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [9]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("/content/My_AxiOraa_Ltd_Synthetic_Annual_Report_2025.pdf")

In [10]:
document=loader.load()
report=""
for page in document:
  report+=page.page_content+'\n'

print("Total Pages:",len(document))

print(report[:1000]) #first 1000 tokens from the input PDF

Total Pages: 9
AxiOraa Ltd.
Synthetic Annual Report 2025
About the Company
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable

In [11]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings #using embedding model specific to Gemini
embedding=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)

In [13]:
from langchain_community.vectorstores import FAISS #FAISS will be the VectorDB for the RAG pipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter #will be helpful during chunking

In [15]:
#chunking on the PDF Data
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [16]:
chunks=text_splitter.split_documents(document)

In [17]:
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embedding #embedding will be performed using a specific embedding model - gemini-embedding-2
)

In [18]:
query="What investment did AxiOraa make in Artificial Intelligence?"

In [19]:
result=vector_store.similarity_search_with_score(
    query=query,
    k=2 #number of possible outcomes
)

for i, (doc, score) in enumerate(result,start=1):
  print(f"Result{i}")
  print(f"Similarity Score: {score}")
  print(f"Page Number: {doc.metadata['page']}")
  print(doc.page_content[:1000])

Result1
Similarity Score: 0.5441054701805115
Page Number: 0
AxiOraa Ltd.
Synthetic Annual Report 2025
About the Company
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark obs

In [21]:
retriever=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":2}
)

In [29]:
rag_prompt=ChatPromptTemplate.from_template(
    """
    You are a Senior Consultant, answer the user's question only by using the retrieved content
    If the answer cannot be found in the context, say: "I couldn't find this information"

    Retrieved Context {context}
    Question {question}

    Provide:
    1. Answer
    2. Supporting Evidence
    3. Page Number
    """)

In [30]:
rag_chain=(rag_prompt | llm | StrOutputParser())

In [31]:
question="Summarize the AI Investments made by AxiOraa"

retrieved_docs=retriever.invoke(question)

context="\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

response=rag_chain.invoke({
    "context":context,
    "question":question
    }
)

print(response)

1.  **Answer:** AxiOraa Ltd. made significant investments in LLMOps, vector databases, evaluation frameworks, LangChain accelerators, and AI observability.

2.  **Supporting Evidence:** "Significant investments were made in LLMOps, vector databases, evaluation frameworks, LangChain accelerators and AI observability."

3.  **Page Number:** I couldn't find this information.


- Q. How did the revenue perform during FY2025?
- Q. What risks were identified in the annual report?